# Fraud detection – logistic + LightGBM + CatBoost + TabPFN blend
Load → preprocess → build models → cross-validate → write `submission.csv`. Run the cells top to bottom.
`prediction` is a fraud score in [0,1]: its ranking gives PR-AUC, and scores ≥ 0.5 are the fraud flags.

In [ ]:
%pip install -q tabpfn lightgbm catboost

import os, sys
import numpy as np
import pandas as pd

# SECURITY: never paste the token into the notebook. In Colab, add it under the key icon (Secrets) as TABPFN_TOKEN.
if "google.colab" in sys.modules:
    from google.colab import userdata
    os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

# Read from the repo's data/ folder if it exists; otherwise open an upload button (Colab)
def load_csv(name):
    for path in [f"../data/{name}", f"data/{name}", name]:
        if os.path.exists(path):
            return pd.read_csv(path)
    from google.colab import files
    print(f"Upload: {name}")
    return pd.read_csv(list(files.upload().keys())[0])

df = load_csv("Track 2 Training Dataset.csv")
test_df = load_csv("Track 2 Testing Dataset.csv")
print("Train:", df.shape, " Test:", test_df.shape)

In [ ]:
target_column = "fraud"
CATS = ["merchant_category", "country", "transaction_channel"]

# Adds per-row features (uses only that row's own values, so no label or cross-row leakage)
def engineer(d):
    d = d.copy()
    d["log_amt"] = np.log1p(d.transaction_amount)
    d["log_spend"] = np.log1p(d.spend_last_24h)
    d["log_age"] = np.log1p(d.account_age)
    d["hour_sin"] = np.sin(2 * np.pi * d.transaction_hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d.transaction_hour / 24)
    d["night"] = d.transaction_hour.isin([23, 0, 1, 2, 3, 4]).astype(int)
    d["burst"] = (d.transactions_last_1h >= 4).astype(float)
    d["high24"] = (d.transactions_last_24h >= 11).astype(float)
    d["young"] = (d.account_age <= 30).astype(float)
    d["amt_to_avg_ticket"] = d.log_amt - np.log1p(d.spend_last_24h / d.transactions_last_24h.clip(lower=1))
    d["share_1h"] = d.transactions_last_1h / d.transactions_last_24h.clip(lower=1)
    d["risky_merchant"] = d.merchant_category.isin(["luxury", "cash_transfer", "electronics"]).astype(int)
    d["night_x_device"] = d.night * d.new_device
    d["n_flags"] = d[["burst", "high24", "young", "new_device", "night", "risky_merchant"]].fillna(0).sum(axis=1)
    for c in CATS:
        d[c] = d[c].fillna("Unknown").astype("category")
    return d

# --- DROP UNNECESSARY ID COLUMNS --- (id is a random label; checked: it carries no fraud signal)
X = engineer(df.drop(columns=["id", target_column]))
y = df[target_column].values
NUM_COLS = [c for c in X.columns if c not in CATS]

# Test set: same features, and exactly the same category lists as training
X_test = engineer(test_df.drop(columns=["id"]))
for c in CATS:
    X_test[c] = pd.Categorical(X_test[c].astype(str), categories=X[c].cat.categories)
X_test = X_test[X.columns]
print(f"{X.shape[1]} features, fraud rate {y.mean():.2%}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.preprocessing import FunctionTransformer
from tabpfn import TabPFNClassifier

# Logistic uses a lean, de-duplicated feature subset (no near-copies, no missing flags): +0.004 blend PR-AUC on 3/3 CV seeds.
# LightGBM and TabPFN still get all 24 features.
LOGIT_COLS = ["transactions_last_24h", "new_device", "log_amt", "log_age", "night", "burst", "high24", "night_x_device"]

def make_logit():
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), LOGIT_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATS),
    ])
    return make_pipeline(pre, LogisticRegression(C=0.3, max_iter=3000))

def make_lgb():
    # Small, heavily regularised LightGBM (handles NaN and pandas categories natively).
    # Tuned by random search (70 configs, 5x2 CV) and confirmed on fresh splits: OOF PR-AUC 0.219 vs 0.207 before.
    # Tiny trees (4 leaves, >=150 rows each) can't memorise single frauds; unweighted (scale_pos_weight=1) was best.
    return lgb.LGBMClassifier(n_estimators=900, learning_rate=0.03, num_leaves=4, min_child_samples=150,
                              colsample_bytree=0.85, reg_lambda=10, reg_alpha=0.5, min_split_gain=0.01,
                              cat_smooth=100, min_data_per_group=50, random_state=0, verbose=-1)

def make_catboost():
    # Extra blend member with a different bias (ordered target statistics for categoricals): +0.005 blend PR-AUC on 3/3 CV seeds.
    # CatBoost needs categoricals as strings.
    to_str = FunctionTransformer(lambda d: d.astype({c: str for c in CATS}))
    return make_pipeline(to_str, CatBoostClassifier(iterations=800, depth=4, learning_rate=0.03, l2_leaf_reg=10,
                                                    cat_features=CATS, random_seed=0, verbose=0))

def make_tabpfn():
    return TabPFNClassifier(ignore_pretraining_limits=True)

MODELS = {"logistic": make_logit, "lightgbm": make_lgb, "catboost": make_catboost, "tabpfn": make_tabpfn}

# Probability of the fraud class
def p_fraud(model, X_):
    return model.predict_proba(X_)[:, list(model.classes_).index(1)]

# Rank-average blend: ranks make the models' different probability scales comparable
def blend(scores):
    return sum(pd.Series(s).rank(pct=True).values for s in scores.values()) / len(scores)

In [ ]:
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

# 5-fold CV: every row is scored once by models that never saw it (out-of-fold)
oof = {name: np.zeros(len(y)) for name in MODELS}
for fold, (tr, te) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(X, y)):
    for name, make in MODELS.items():
        oof[name][te] = p_fraud(make().fit(X.iloc[tr], y[tr]), X.iloc[te])
    print(f"fold {fold} done")
oof["blend"] = blend({m: oof[m] for m in MODELS})

for name, p in oof.items():
    print(f"{name:9s} PR-AUC = {average_precision_score(y, p):.4f}")

# Threshold: PR-AUC does not depend on it (it only uses the ranking). It sets the 0/1 fraud flags scored by F1 / Recall.
# For every candidate cut-off (share of rows flagged), build the confusion matrix and compute precision, recall and F1.
rows = []
for share in np.arange(0.002, 0.0601, 0.0005):
    flag = (oof["blend"] >= np.quantile(oof["blend"], 1 - share)).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, flag).ravel()
    rows.append({"share": share, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                 "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn), "F1": 2 * tp / (2 * tp + fp + fn)})
cm = pd.DataFrame(rows)
cm["F1_smooth"] = cm["F1"].rolling(5, center=True, min_periods=1).mean()   # don't pick a one-off noisy spike
best = cm.loc[cm["F1_smooth"].idxmax()]
best_share = best["share"]

print(f"Flag top {best_share:.2%} of rows: precision {best.precision:.3f}, recall {best.recall:.3f}, F1 {best.F1:.3f}")
print(pd.DataFrame([[int(best.TN), int(best.FP)], [int(best.FN), int(best.TP)]],
                   index=["actual 0", "actual 1"], columns=["predicted 0", "predicted 1"]))


In [ ]:
# Maps scores to [0,1] without changing their order (PR-AUC unchanged) and puts the flag threshold at 0.5
def to_submission_score(score, thr):
    score = np.asarray(score, dtype=float)
    lo, hi = score.min(), score.max()
    below = 0.49 * (score - lo) / max(thr - lo, 1e-12)
    above = 0.51 + 0.49 * (score - thr) / max(hi - thr, 1e-12)
    return np.clip(np.where(score >= thr, above, below), 0, 1)

# Train every model on all 20,000 training rows, score the test set, blend
test_scores = {name: p_fraud(make().fit(X, y), X_test) for name, make in MODELS.items()}
test_blend = blend(test_scores)
test_prob = to_submission_score(test_blend, np.quantile(test_blend, 1 - best_share))

submission = pd.DataFrame({"id": test_df["id"], "prediction": test_prob})
assert len(submission) == len(test_df) and submission["id"].is_unique
assert submission["prediction"].between(0, 1).all() and submission["prediction"].nunique() > 2
submission.to_csv("submission.csv", index=False)
print(f"submission.csv written: {len(submission)} rows, flagged (>= 0.5): {(test_prob >= 0.5).sum()}")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download("submission.csv")